In [ ]:
'''
Name That LLM | Iteration 01
Runnable OOP word/character TF-IDF baseline trained on official train.csv. This is a lightweight baseline, not a reproduction of the public 0.55 ensemble.

Competition data: https://www.kaggle.com/competitions/name-that-llm
Method inspiration only (not reproduced): Lucas Boesen, https://www.kaggle.com/code/lucasboesen/lora-x-adaptive-centering
'''
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

@dataclass(frozen=True)
class CompetitionConfig:
    input_dir: Path = Path('/kaggle/input/competitions/name-that-llm')
    output_path: Path = Path('/kaggle/working/submission.parquet')
    labels: tuple = ('gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3')
    claude_labels: tuple = ('claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5')
    alpha: float = 0.5



In [ ]:
'''
Load and validate the official competition files.
'''
class CompetitionData:
    def __init__(self, config): self.config = config
    def load(self):
        train = pd.read_csv(self.config.input_dir / 'train.csv')
        test = pd.read_csv(self.config.input_dir / 'test.csv')
        if not {'id','text','model'}.issubset(train) or not {'id','text'}.issubset(test): raise ValueError('Unexpected data schema')
        for name, frame in [('train',train),('test',test)]:
            if frame.id.isna().any() or frame.id.duplicated().any() or frame.text.isna().any(): raise ValueError(f'Invalid IDs or text in {name}')
        if set(train.model) != set(self.config.labels): raise ValueError('train.csv must contain all 18 competition labels')
        return train, test


In [ ]:
'''
Build word and character TF-IDF features, then train the text classifier.
'''
class TfidfModel:
    def __init__(self, labels):
        self.labels = tuple(labels)
        self.word = TfidfVectorizer(analyzer='word',ngram_range=(1,2),min_df=2,max_features=80000,sublinear_tf=True,strip_accents='unicode')
        self.char = TfidfVectorizer(analyzer='char',ngram_range=(3,5),min_df=2,max_features=180000,sublinear_tf=True,dtype=np.float32)
        self.clf = LogisticRegression(class_weight='balanced',max_iter=500,C=4.0)
    def fit(self, texts, targets):
        texts = pd.Series(texts).astype(str)
        x = hstack([self.word.fit_transform(texts),self.char.fit_transform(texts)],format='csr',dtype=np.float32)
        self.clf.fit(x,targets)
        if set(self.clf.classes_) != set(self.labels): raise ValueError('Classifier is missing a target class')
        return self
    def predict_proba(self, texts):
        texts = pd.Series(texts).astype(str)
        x = hstack([self.word.transform(texts),self.char.transform(texts)],format='csr',dtype=np.float32)
        order = [list(self.clf.classes_).index(k) for k in self.labels]
        return self.clf.predict_proba(x)[:,order]


In [ ]:
'''
Train the baseline classifier and predict all 18 model probabilities.
'''
class BaselineExperiment:
    def __init__(self, config): self.config = config
    def run(self, train, test):
        return TfidfModel(self.config.labels).fit(train.text,train.model).predict_proba(test.text)


In [ ]:
'''
Validate predictions and write the required submission.parquet file.
'''
class SubmissionBuilder:
    def __init__(self, config): self.config = config
    def save(self, test, probabilities):
        p = np.asarray(probabilities)
        if p.shape != (len(test),len(self.config.labels)) or not np.isfinite(p).all(): raise ValueError('Invalid prediction matrix')
        if (p < 0).any() or not np.allclose(p.sum(axis=1),1,atol=1e-6): raise ValueError('Invalid probability rows')
        pred = [self.config.labels[i] for i in p.argmax(axis=1)]
        out = pd.DataFrame({'id':test.id.to_numpy(),'model':pred})
        if out.isna().any().any() or out.id.duplicated().any() or not set(pred) <= set(self.config.labels): raise ValueError('Invalid submission')
        self.config.output_path.parent.mkdir(parents=True,exist_ok=True)
        out.to_parquet(self.config.output_path,index=False)
        return out


In [ ]:
'''
Run the experiment from data loading through submission creation.
'''
config=CompetitionConfig()
train,test=CompetitionData(config).load()
probabilities=BaselineExperiment(config).run(train,test)
submission=SubmissionBuilder(config).save(test,probabilities)
print(f'Wrote {len(submission):,} rows to {config.output_path}')
submission.head()
